## Example: series approximations

N.B. This example is only fun if you really like maths. 

We can approximate the value of $e^x$ using the series $$\sum_{i = 0}^{\infty} \frac{x^i}{i!}$$


We will use numpy to compare partial sums $\sum_{i = 0}^{n} \frac{x^i}{i!}$ for increasing values of $n$ to the correct value.

The function ``vectorize`` allows a non-numpy function to be applied to all elements in an array, we use this to calculate $n!$, which is used within the patial sum approximation.

In [4]:
import math
import numpy as np

fac = np.vectorize(math.factorial) 

def approx_exp(x, num=10):
    n = np.arange(num)
    return np.sum((x ** n) / fac(n))

The following cells break out this function into its parts so we can see what's happening. First decide the range of values we will calculate the partial sum over.

In [ ]:
n = np.arange(10)
n

Suppose x=2. We calculate $x^i$ and divide this by $i!$ for each value $i$ in the array ``n``.

In [ ]:
p = (2 ** n)
p

In [ ]:
f = fac(n)
f

In [ ]:
p / f

Finally, take the sum.

In [ ]:
np.sum(p / f)

And this is exactly what the function `approx_exp` above does. 

In [ ]:
approx_exp(2, 10) # approximate $e^2$

In [ ]:
approx_exp(3, 10) # approximate $e^3$

The following function allows us to see how the approximation improves as we take the partial sum over more values. 

In [5]:
def pretty_print_exp(x, num):
    print("Actual:", math.e ** x) 

    print("i (terms)\tPartial Sum\tError")

    for i in range(1, num+1):
        approx_val = approx_exp(x, num=i)
        print(f"{i}\t\t{approx_val:.03f}\t\t{math.e**x - approx_val:.03f}")

In [6]:
pretty_print_exp(3, 10)

Actual: 20.085536923187664
i (terms)	Partial Sum	Error
1		1.000		19.086
2		4.000		16.086
3		8.500		11.586
4		13.000		7.086
5		16.375		3.711
6		18.400		1.686
7		19.412		0.673
8		19.846		0.239
9		20.009		0.076
10		20.063		0.022


In [ ]:
pretty_print_exp(4, 10)

We'll try to generalise this so we can pass in different approximations and work on multiple input values (x) at once. We can also see that some calculation is repeated within the python ``for`` loop because calculating a partial sum to $n+1$ necessarily involves also calculating the sum to $n$. We can use numpy to avoid this repeated effort.

In [7]:
input_vals = np.array([3,4])
input_vals

array([3, 4])

In [8]:
n = np.arange(5)
n

array([0, 1, 2, 3, 4])

We want to use these two arrays together but they have different shapes, so we'll need to reshape in order to allow numpy to use broadcasting. 

In [10]:
input_vals = input_vals.reshape(2,1)
n = n.reshape(1,5)

Calculate the powers $3^i$ and $4^i$ in one array.

In [11]:
p = np.power(input_vals, n)
p

array([[  1,   3,   9,  27,  81],
       [  1,   4,  16,  64, 256]], dtype=int32)

As before, we divide each $x^i$ by $i!$ using the vectorized factorial function. 

In [12]:
d = np.true_divide(p, fac(n))
d

array([[ 1.        ,  3.        ,  4.5       ,  4.5       ,  3.375     ],
       [ 1.        ,  4.        ,  8.        , 10.66666667, 10.66666667]])

And take the sum, note the use of the axis parameter.

In [13]:
np.sum(d, axis = 1)

array([16.375     , 34.33333333])

To reduce the work within the python for loop, we can instead use numpy's cumulative sum function.

In [14]:
np.cumsum(d, axis = 1) # get cumulative sum to each position

array([[ 1.        ,  4.        ,  8.5       , 13.        , 16.375     ],
       [ 1.        ,  5.        , 13.        , 23.66666667, 34.33333333]])

Putting the previous several cells together:

In [15]:
approx_exp = np.cumsum(np.true_divide(np.power(input_vals, n), fac(n)), axis = 1)

In [16]:
approx_exp

array([[ 1.        ,  4.        ,  8.5       , 13.        , 16.375     ],
       [ 1.        ,  5.        , 13.        , 23.66666667, 34.33333333]])

Alternatively as a lambda

In [17]:
approx_exp = lambda x,n: np.cumsum(np.true_divide(np.power(x,n), fac(n)), axis = 1)

In [18]:
approx_exp(input_vals, n)

array([[ 1.        ,  4.        ,  8.5       , 13.        , 16.375     ],
       [ 1.        ,  5.        , 13.        , 23.66666667, 34.33333333]])

Or just as a normal function -- probably more readable and we can put in some validation. 

In [19]:
def approx_exp(x, n):
    assert(x.ndim == 2)
    assert(x.shape[1] == 1)
    assert(n.ndim == 2)
    assert(n.shape[0] == 1)
    
    fac = np.vectorize(math.factorial)
    
    return np.cumsum(np.true_divide(np.power(x,n), fac(n)), axis = 1) 

In [20]:
approx_exp(input_vals, n)

array([[ 1.        ,  4.        ,  8.5       , 13.        , 16.375     ],
       [ 1.        ,  5.        , 13.        , 23.66666667, 34.33333333]])

Now we have the function for estimating $e^x$ we can abstract further: create a function which takes the estimation function as a parameter, as well as the different arguments for it. Later we can calculate different partial sums using the same framework. 

In [ ]:
def calculate_partial_series(series_func, input_values, num = 10):
    n = np.arange(num).reshape(1,-1) # using -1 in reshape allows numpy to decide best value
    return series_func(input_values, n)

In [ ]:
calculate_partial_series(approx_exp, input_vals, 5)

In [ ]:
calculate_partial_series(approx_exp, input_vals)

Previously we compared the partial sums to the real value. We need multiple target values if we're calcualting multiple series at once.

In [ ]:
targets = np.exp(input_vals)
targets

In [ ]:
targets.shape

We will repeat these target values so we can easily calucate the difference between each partial sum and the target.

In [ ]:
num = 10
np.full([targets.shape[0], num], targets)

In [ ]:
def compare_partial_series(targets, series_func, input_values, num = 10):    
    assert(targets.ndim == 2)
    assert(targets.shape[1] == 1)
    
    true_outputs = np.full([targets.shape[0], num], targets)

    approx_outputs = calculate_partial_series(series_func, input_values, num)
    
    diffs = true_outputs - approx_outputs
    
    return approx_outputs, diffs 

In [ ]:
targets

In [ ]:
input_vals

In [ ]:
approx_values, diffs = compare_partial_series(targets, approx_exp, input_vals)

In [ ]:
diffs

It's a bit annoying that we have to pass in the targets and input values already as the right shape, so let's allow for them to be passed in as ordinary lists.

In [ ]:
def compare_partial_series(targets, series_func, input_values, num = 10):    
    # allow to pass in lists of inputs / targets instead
    if isinstance(input_values, list):
        input_values = np.array(input_values)
    if input_values.ndim == 1:
        input_values = input_values.reshape(-1,1)
    
    if isinstance(targets, list):
        targets = np.array(targets)
    if targets.ndim == 1:
        targets = targets.reshape(-1,1)
    
    assert(targets.ndim == 2)
    assert(targets.shape[1] == 1)
    
    true_outputs = np.full([targets.shape[0], num], targets)

    approx_outputs = calculate_partial_series(series_func, input_values, num)
    
    diffs = true_outputs - approx_outputs
    
    return approx_outputs, diffs

In [ ]:
approx_values, diffs = compare_partial_series([20.08553692, 54.59815003], approx_exp, [3,4])

In [ ]:
diffs

We could even pass in the true function and calculate the targets locally. 

In [ ]:
def compare_partial_series(actual_func, series_func, input_values, num = 10):        
    # allow to pass in lists of inputs instead of array
    if isinstance(input_values, list):
        input_values = np.array(input_values)
    if input_values.ndim == 1:
        input_values = input_values.reshape(-1,1)
    
    targets = actual_func(input_values)
    true_outputs = np.full([targets.shape[0], num], targets)

    approx_outputs = calculate_partial_series(series_func, input_values, num)
    
    diffs = true_outputs - approx_outputs
    
    return targets, approx_outputs, diffs

In [ ]:
comp_out = compare_partial_series(np.exp, approx_exp, [3,4])

In [ ]:
comp_out

This function has given us all the data we need, now we'll try to present it nicely. We first extract the partial sums and errors for one of the inputs, then we reshape so that the approximate value and error are together. 

In [ ]:
np.array(comp_out[1:])[:,0]

In [ ]:
np.swapaxes(np.array(comp_out[1:])[:,0], 0, 1)

In [ ]:
def pretty_print_approx_func(actual_func, series_func, input_values, num = 10):
    comp_output = compare_partial_series(actual_func, series_func, input_values)
    targets = comp_output[0]
    estimates_and_diffs = np.array(comp_output[1:])
    
    for i, t in enumerate(targets):
        t = t[0]
        print("Input:", input_values[i])
        print("Actual:", t) 
        print("i (terms)\tPartial Sum\t\tError")
        values = np.swapaxes(estimates_and_diffs[:,i], 0, 1)

        for j, val in enumerate(values):
            print(f"{j}\t\t{val[0]:.06f}\t\t{val[1]:.06f}")
        
        print("")

In [ ]:
pretty_print_approx_func(np.exp, approx_exp, [3,4])

Now we can use the same framework with other partial sums.
$$\sum_{i = 0}^{\infty} \frac{1}{2^i} = 2$$
and more generally
$$\sum_{i = 0}^{\infty} x^i = \frac{1}{1-x}$$
for $$|x| < 1$$

In [ ]:
def geom(x, n):
    assert(x.ndim == 2)
    assert(x.shape[1] == 1)
    assert(n.ndim == 2)
    assert(n.shape[0] == 1)
    
    return np.cumsum(np.power(x,n), axis = 1) 

In [ ]:
def geom_limit(x):
    assert(x > -1 and x < 1 and x != 0)
    return 1/(1-x)

geom_limit_vec = np.vectorize(geom_limit)

Test with $x=0.5$

In [ ]:
geom(np.array([0.5]).reshape(1,1), np.arange(5).reshape(1,-1))

In [ ]:
compare_partial_series(geom_limit_vec, geom, [0.5])

In [ ]:
pretty_print_approx_func(geom_limit_vec, geom, [0.5], num = 10)

Test with multiple values

In [ ]:
input_vals = [0.5, 0.25, 0.3]

pretty_print_approx_func(geom_limit_vec, geom, input_vals)